In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abderhmanahmed/lung-disease-data/Healthcare.csv
/kaggle/input/datasets/abderhmanahmed/lung-disease-data/respiratory symptoms and treatment.csv
/kaggle/input/datasets/abderhmanahmed/lung-disease-data/Symptom2Disease.csv
/kaggle/input/models/abderhmanahmed/lung-disease/transformers/default/1/config.json
/kaggle/input/models/abderhmanahmed/lung-disease/transformers/default/1/training_args.bin
/kaggle/input/models/abderhmanahmed/lung-disease/transformers/default/1/tokenizer.json
/kaggle/input/models/abderhmanahmed/lung-disease/transformers/default/1/tokenizer_config.json
/kaggle/input/models/abderhmanahmed/lung-disease/transformers/default/1/model.safetensors


In [ ]:
import pandas as pd

data1 = pd.read_csv('Dataset/Healthcare.csv')
data2 = pd.read_csv('Dataset/respiratory symptoms and treatment.csv')
data3 = pd.read_csv('Dataset/Symptom2Disease.csv')
0
print('Healthcare.csv Head:')
print(data1.shape)
print(data1.head())
print('\nrespiratory symptoms and treatment.csv Head:')
print(data2.shape)
print(data2.head())
print('\nSymptom2Disease.csv Head:')
print(data3.shape)
print(data3.head())

Healthcare.csv Head:
(25000, 6)
   Patient_ID  Age  Gender                                           Symptoms  \
0           1   29    Male              fever, back pain, shortness of breath   
1           2   76  Female                   insomnia, back pain, weight loss   
2           3   78    Male                    sore throat, vomiting, diarrhea   
3           4   58   Other  blurred vision, depression, weight loss, muscl...   
4           5   55  Female                    swelling, appetite loss, nausea   

   Symptom_Count           Disease  
0              3           Allergy  
1              3  Thyroid Disorder  
2              3         Influenza  
3              4            Stroke  
4              3     Heart Disease  

respiratory symptoms and treatment.csv Head:
(38537, 6)
                     Symptoms  Age     Sex Disease    Treatment Nature
0                   coughing   5.0  female  Asthma   Omalizumab   high
1  tight feeling in the chest  4.0  female  Asthma  Mepolizu

In [3]:
print('data1 shape final: ',data1.loc[data1['Disease'].isin(data2['Disease'].unique())].shape)
print('data3 shape final: ',data3.loc[data3['label'].isin(data2['Disease'].unique())].shape)


data1 shape final:  (3250, 6)
data3 shape final:  (50, 3)


In [4]:
df_symptoms_data1 = data1[['Symptoms', 'Disease']]
df_symptoms_data2 = data2[['Symptoms', 'Disease']]
df_symptoms_data3 = data3.rename(columns={'text': 'Symptoms', 'label': 'Disease'})[['Symptoms', 'Disease']]

# Concatenate all three dataframes
symptoms_df = pd.concat([df_symptoms_data1, df_symptoms_data2, df_symptoms_data3], ignore_index=True)
#symptoms_df = pd.concat([df_symptoms_data2], ignore_index=True)


# Filter symptoms_df to include only diseases present in data2
symptoms_df = symptoms_df[symptoms_df['Disease'].isin(data2['Disease'].unique())]


print("Combined Symptoms and Disease DataFrame shape (filtered by data2 diseases):", symptoms_df.shape)
display(symptoms_df.head(5))

Combined Symptoms and Disease DataFrame shape (filtered by data2 diseases): (41837, 2)


,Symptoms,Disease
2,"sore throat, vomiting, diarrhea",Influenza
23,"weight gain, sore throat, back pain, sneezing",Influenza
28,"cough, tremors, dizziness",Asthma
34,"sweating, fever, sore throat, swelling, diarrh...",Influenza
41,"runny nose, dizziness, swelling, sore throat, ...",Tuberculosis


In [5]:
print(symptoms_df['Disease'].unique())

['Influenza' 'Asthma' 'Tuberculosis' 'Pneumonia' nan 'Bronchiectasis'
 'bronchiolitis' 'bronchitis' 'Chronic Bronchitis' 'Chronic cough'
 'chronic obstructive pulmonary disease' 'Mesothelioma' 'Pneumothorax'
 'Pulmonary hypertension' 'sleep apnea' 'Respiratory syncytial virus'
 'Acute Respiratory Distress Syndrome' 'Asbestosis' 'Aspergillosis']


In [6]:
symptoms_df.to_csv("/kaggle/working//dataset.csv", index=False)

In [7]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

symptoms_df["Label"] = encoder.fit_transform(symptoms_df["Disease"])

print(symptoms_df.head())

                                             Symptoms       Disease  Label
2                     sore throat, vomiting, diarrhea     Influenza      7
23      weight gain, sore throat, back pain, sneezing     Influenza      7
28                          cough, tremors, dizziness        Asthma      3
34  sweating, fever, sore throat, swelling, diarrh...     Influenza      7
41  runny nose, dizziness, swelling, sore throat, ...  Tuberculosis     13


In [8]:
import joblib

joblib.dump(encoder, "/kaggle/working//label_encoder.pkl")

['/kaggle/working//label_encoder.pkl']

In [9]:
from sklearn.model_selection import train_test_split

# Drop rows with any NaN values from symptoms_df before splitting
symptoms_df_cleaned = symptoms_df.dropna(subset=['Symptoms', 'Disease'])

# Assuming symptoms_df_cleaned is your DataFrame and 'Disease' is your target variable
X = symptoms_df_cleaned['Symptoms']
y = symptoms_df_cleaned['Label']

# Split into 70% training and 30% temporary (for validation and test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# Split temporary into 15% validation and 15% test (50% of temporary for each)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Training set size: {len(X_train)} ({len(X_train)/len(symptoms_df_cleaned):.2%})")
print(f"Validation set size: {len(X_val)} ({len(X_val)/len(symptoms_df_cleaned):.2%})")
print(f"Testing set size: {len(X_test)} ({len(X_test)/len(symptoms_df_cleaned):.2%})")

Training set size: 28695 (70.00%)
Validation set size: 6149 (15.00%)
Testing set size: 6149 (15.00%)


In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    X_val.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
import torch

class DiseaseDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {}

        for key, value in self.encodings.items():

            item[key] = torch.tensor(value[idx])

        item["labels"] = torch.tensor(
            self.labels.iloc[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

In [12]:
train_dataset = DiseaseDataset(
    train_encodings,
    y_train
)

test_dataset = DiseaseDataset(
    test_encodings,
    y_val
)

In [13]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(encoder.classes_)
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=5e-5,

    per_device_train_batch_size=32,

    per_device_eval_batch_size=32,

    num_train_epochs=9,

    weight_decay=0.01,

    logging_steps=10,

    load_best_model_at_end=True
)

In [15]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [16]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [17]:
from transformers import Trainer

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [18]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.828262,1.801447,0.617174,0.718090,0.617174,0.621783
2,1.727309,1.720838,0.625955,0.704432,0.625955,0.617478
3,1.691609,1.711753,0.615547,0.663401,0.615547,0.595745
4,1.713959,1.700611,0.614571,0.683832,0.614571,0.616751
5,1.547416,1.698461,0.616360,0.665747,0.616360,0.602577
6,1.679616,1.695329,0.620263,0.644440,0.620263,0.600314
7,1.751670,1.684253,0.626606,0.657150,0.626606,0.597618
8,1.739393,1.696529,0.625630,0.656842,0.625630,0.598170
9,1.578886,1.710616,0.625630,0.667155,0.625630,0.605851


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=4041, training_loss=1.760016016988464, metrics={'train_runtime': 1909.5641, 'train_samples_per_second': 135.243, 'train_steps_per_second': 2.116, 'total_flos': 8627749252498350.0, 'train_loss': 1.760016016988464, 'epoch': 9.0})

In [19]:
#trainer.save_model("lung_model_data2only")
#tokenizer.save_pretrained("lung_model_data2only")

In [20]:
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
import torch.nn.functional as F
import numpy as np

# Define a simple Dataset for inference
class InferenceDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

# Tokenize X_test
inputs_test = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=64,
    return_tensors="pt"
)

# Create a dataset and DataLoader for X_test, including y_test labels
inference_dataset = InferenceDataset(inputs_test, y_test)

# Adjust batch size based on GPU memory. Start with a conservative value.
batch_size = 32 # You might need to adjust this value
inference_loader = DataLoader(inference_dataset, batch_size=batch_size, shuffle=False)

# Move model to appropriate device (CPU or GPU)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

model.eval()

all_predictions = []
all_true_labels = []
all_logits = []

with torch.no_grad():
    for batch in inference_loader:
        # Move input tensors and labels for the current batch to the same device as the model
        batch_inputs = {key: value.to(device) for key, value in batch.items() if key != 'labels'}
        batch_labels = batch['labels'].to(device) # Keep labels on the device for loss calculation

        outputs = model(**batch_inputs)
        logits = outputs.logits
        prediction = logits.argmax(-1)

        all_predictions.extend(prediction.cpu().numpy())
        all_true_labels.extend(batch_labels.cpu().numpy())
        all_logits.append(logits.cpu()) # Collect logits from CPU to avoid OOM if all_logits become too large

# Convert collected lists to numpy arrays
final_predictions = np.array(all_predictions)
final_true_labels = np.array(all_true_labels)
final_logits = torch.cat(all_logits, dim=0)

# Calculate accuracy
accuracy = accuracy_score(final_true_labels, final_predictions)

# Calculate Cross-Entropy Loss
# F.cross_entropy expects logits and true labels
# Make sure final_true_labels is a long tensor for cross_entropy
loss = F.cross_entropy(final_logits, torch.tensor(final_true_labels, dtype=torch.long))

print(f"Accuracy on X_test: {accuracy:.4f}")
print(f"Cross-Entropy Loss on X_test: {loss.item():.4f}")

print("\nExample predictions (first 10) and true labels:")
for i in range(10):
    predicted_disease = encoder.inverse_transform([final_predictions[i]])[0]
    true_disease = encoder.inverse_transform([final_true_labels[i]])[0]
    print(f"Sample {i+1}: Predicted='{predicted_disease}', True='{true_disease}'")


/tmp/ipykernel_58/3861418436.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Accuracy on X_test: 0.6264
Cross-Entropy Loss on X_test: 0.8596

Example predictions (first 10) and true labels:
Sample 1: Predicted='Pneumonia', True='Pneumonia'
Sample 2: Predicted='chronic obstructive pulmonary disease', True='chronic obstructive pulmonary disease'
Sample 3: Predicted='Mesothelioma', True='Asthma'
Sample 4: Predicted='Influenza', True='Influenza'
Sample 5: Predicted='Pneumonia', True='Pneumonia'
Sample 6: Predicted='Mesothelioma', True='Asthma'
Sample 7: Predicted='chronic obstructive pulmonary disease', True='chronic obstructive pulmonary disease'
Sample 8: Predicted='sleep apnea', True='sleep apnea'
Sample 9: Predicted='Mesothelioma', True='Chronic Bronchitis'
Sample 10: Predicted='Pneumonia', True='Influenza'


In [21]:
probabilities = torch.softmax(
    outputs.logits,
    dim=1
)

print(probabilities)

tensor([[1.4947e-04, 9.6982e-05, 2.9354e-04, 1.8061e-01, 7.8769e-04, 2.8760e-04,
         5.1215e-04, 2.6226e-01, 3.1347e-04, 3.6520e-01, 9.5319e-04, 6.0430e-04,
         6.0628e-05, 1.8530e-01, 1.1206e-03, 6.8449e-04, 6.2302e-04, 9.4030e-05,
         4.7057e-05],
        [7.6659e-05, 5.3429e-05, 3.1196e-05, 1.8933e-05, 5.1198e-05, 2.0946e-05,
         1.1265e-05, 3.6135e-05, 1.6046e-05, 3.4295e-06, 1.9737e-06, 1.2802e-04,
         9.4083e-05, 7.0195e-05, 2.8848e-05, 2.3134e-05, 7.6910e-06, 9.9876e-01,
         5.6412e-04],
        [9.9602e-01, 6.1777e-04, 1.7419e-04, 2.8518e-05, 8.1049e-05, 2.7279e-05,
         1.8836e-04, 2.7345e-04, 4.8524e-05, 7.9453e-06, 7.3049e-05, 2.3211e-04,
         1.9049e-04, 1.1663e-04, 5.6938e-04, 4.8612e-05, 6.4186e-05, 4.4468e-04,
         7.9183e-04],
        [5.7336e-04, 4.2203e-05, 9.4373e-05, 2.4721e-04, 2.7644e-04, 1.2707e-04,
         1.3072e-04, 2.3669e-03, 8.2522e-05, 1.0255e-03, 7.6816e-05, 6.7768e-05,
         2.6652e-04, 1.4060e-04, 5.7139e-01